# ContractRisk — corrección 02 de integración

Sesión nueva de Colab con GPU T4. Ejecutar en orden. No modifica el Space actual ni usa el gold final.

Prueba transcripción visual + extracción determinista de campos, control de imágenes uniformes y elegibilidad de la herramienta. Reutiliza las cuatro imágenes originales con hashes verificados, no repite M1 y carga el generador una sola vez después de liberar el VLM.

La orquestación visual reutiliza las lecturas guardadas: no es latencia fresca punta a punta ni benchmark de eficiencia. Se distinguen errores, formato válido, control de píxeles y llamadas reales del VLM. Calidad sujeta a revisión humana.

Para W&B: habilitar el secreto WANDB_API_KEY. Sin él se conservan resultados locales. La última celda descarga evidencia incluso si falló la ejecución.

In [1]:
import os, sys, subprocess, tempfile, json
from pathlib import Path

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_final_correction02_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError(f'Proceso terminó con código {status}; consultar {logfile}')

try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
    if key:
        os.environ['WANDB_API_KEY'] = key
    del key
    print('Secreto W&B disponible; no se imprime su valor.')
except Exception:
    print('Sin secreto W&B accesible: la prueba guardará evidencia local.')


Commit: 8cda4e8fae6554447e04c6e2e0b1200035ee5d70
Sin secreto W&B accesible: la prueba guardará evidencia local.


In [2]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r',
    'final/requirements-integration.txt'], 'installation.log')
import torch
run_visible([sys.executable, '-c', 'from final.run_integration import fixture_font_path; print(fixture_font_path().name)'], 'font_check.log')
assert torch.cuda.is_available(), 'Selecciona GPU T4 y una sesión nueva.'
print('GPU:', torch.cuda.get_device_name(0))
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 8.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 25.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 55.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 83.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 88.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 38.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 94.1 MB/s 

In [3]:
RESULTS = ROOT / 'correction02_development_results'
run_visible([sys.executable, '-u', 'final/run_correction02.py', '--output', str(RESULTS)], 'correction02.log')


Loading VLM first; uniform image handled by pixel guard...
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!

Fetching 2 files: 100%|██████████| 2/2 [00:27<00:00, 13.85s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:18<00:00,  9.14s/it]
visual_component visual_dev_1
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
visual_component visual_dev_1_contrast
visual_component visual_dev_2
visual_component visual_blank
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientSta

## Inspección y conservación de resultados

Ejecutar las celdas siguientes también si falla la integración. `quality_approved=false` es deliberado: las lecturas, fuentes y respuestas necesitan revisión humana.

Los eventos de W&B de esta prueba deben quedar separados de la futura evaluación final y finalizar con `finish()`. Si la conexión falla, conservar la evidencia local; no publicar la clave.


In [4]:
RESULTS = ROOT / 'correction02_development_results'
metadata = RESULTS / 'metadata.json'
print(metadata.read_text(encoding='utf-8') if metadata.exists() else 'Sin metadata: revisar installation.log/correction02.log.')
rows_path = RESULTS / 'correction_rows.jsonl'
if rows_path.exists():
    for line in rows_path.read_text(encoding='utf-8').splitlines():
        row = json.loads(line)
        result = row.get('result', {})
        print(row['stage'], row['case_id'], row.get('error', ''),
            json.dumps(result.get('output', result.get('visual', {})), ensure_ascii=False))


{
  "status": "completed_pending_human_review",
  "purpose": "correction02_development_not_final_evaluation",
  "version": "final-integration-dev-0.2",
  "models": {
    "embedding": [
      "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
      "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
    ],
    "generator": [
      "Qwen/Qwen2.5-3B-Instruct",
      "aa8e72537993ba99e69dfaafa59ed015b17504d1"
    ],
    "reranker": [
      "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
      "1427fd652930e4ba29e8149678df786c240d8825"
    ],
    "vision": [
      "Qwen/Qwen2-VL-2B-Instruct",
      "895c3a49bc3fa70a340399125c650a463535e71c"
    ],
    "classifier": [
      "BSC-LT/RoBERTalex",
      "2a1c89fd468a362368463b4126751fdd51c4d847"
    ]
  },
  "seed": 42,
  "started_utc": "2026-11-16T18:46:17.740808+00:00",
  "quality_approved": false,
  "exceptions": [],
  "wandb_status": "disabled",
  "latency_policy": "component VLM separate, then cached-read orchestration; not fresh end-to-e

In [5]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, RESULTS / log.name)
package = shutil.make_archive(str(ROOT / 'contractrisk_final_correccion_02_resultados'), 'zip', RESULTS)
files.download(package)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>